# Pipe-1: Complete End-to-End ML Pipeline
## Synthetic Image Attribution Challenge

**Complete workflow:** Data Loading → EDA → Preprocessing → CV Split → Model Training → Validation → Checkpoint Selection → Inference & Submission

Follow the README.md alongside this notebook for detailed explanations.

## Setup & Configuration

In [ ]:
# Install required packages (if needed)
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'timm',
    'albumentations',
    'pandas',
    'numpy',
    'pillow',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'tomli'  # For TOML config loading (Python < 3.11)
]

print("Checking/installing packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package}")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

print("\n✓ All packages available")

Checking/installing packages...
✓ torch
✓ torchvision
✓ timm
✓ albumentations
✓ pandas
✓ numpy
Installing pillow...
✓ pillow installed
✓ matplotlib
✓ seaborn
Installing scikit-learn...
✓ scikit-learn installed

✓ All packages available


In [ ]:
# EXACT: Import all required libraries
import os
import sys
import json
import csv
import time
import logging
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ All imports successful
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
# EXACT: Setup embedded config and logging (self-contained for Kaggle)
import os
import sys

os.chdir('/kaggle/working')

# ============================================================================
# EMBEDDED CONFIG - All settings in one place (no external files needed)
# ============================================================================
class PipelineConfig:
    # Paths
    working_dir = "/kaggle/working"
    input_dir = "/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data"
    checkpoint_base = "checkpoints"
    final_models = "final_models"
    submission = "submission"
    logs = "logs"
    eda_outputs = "eda_outputs"
    
    # Data
    num_classes = 10
    target_size = 224
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    
    # Training
    model_name = "tf_efficientnetv2_m.in21k_ft_in1k"
    pretrained = True
    epochs = 15
    batch_size_train = 32
    batch_size_val = 32
    learning_rate = 1e-4
    dropout_rate = 0.45
    dropout_path_rate = 0.25
    weight_decay = 1e-5
    label_smoothing = 0.1
    optimizer_betas = (0.9, 0.999)
    optimizer_eps = 1e-8
    scheduler_t_max = 15
    scheduler_eta_min = 1e-6
    gradient_clip_norm = 1.0
    num_folds = 5
    random_seed = 42
    
    # Validation
    top_k_models = 5
    metric = "gen_score"
    
    # Augmentation
    augmentation_rotate = 5
    augmentation_hflip_prob = 0.5
    augmentation_vflip_prob = 0.2
    augmentation_brightness = (-0.2, 0.2)
    augmentation_contrast = (-0.2, 0.2)
    augmentation_brightness_contrast_prob = 0.5
    augmentation_blur_prob = 0.3
    augmentation_crop_prob = 0.1
    
    @classmethod
    def resolve_path(cls, relative_path):
        return Path(cls.working_dir) / relative_path
    
    @classmethod
    def get_checkpoint_dir(cls, fold):
        return cls.resolve_path(cls.checkpoint_base) / f"fold_{fold}"
    
    @classmethod
    def get_final_models_dir(cls):
        return cls.resolve_path(cls.final_models)
    
    @classmethod
    def get_logs_dir(cls):
        return cls.resolve_path(cls.logs)
    
    @classmethod
    def get_submission_dir(cls):
        return cls.resolve_path(cls.submission)
    
    @classmethod
    def get_eda_dir(cls):
        return cls.resolve_path(cls.eda_outputs)

config = PipelineConfig()

# ============================================================================
# EMBEDDED LOGGING - Dynamic logger setup (no external module needed)
# ============================================================================
def setup_pipeline_logger(stage_name="Pipeline"):
    """Setup logger for pipeline stage"""
    log_level = os.environ.get("LOG_LEVEL", "INFO")
    log_level = getattr(logging, log_level.upper(), logging.INFO)
    
    logger = logging.getLogger("pipeline")
    logger.setLevel(log_level)
    
    # Clear existing handlers
    logger.handlers.clear()
    
    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(log_level)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    # File handler
    config.get_logs_dir().mkdir(parents=True, exist_ok=True)
    log_file = config.resolve_path(f"{config.logs}/pipeline.log")
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(log_level)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    logger.info(f"\n{'='*60}")
    logger.info(f"STAGE: {stage_name}")
    logger.info(f"{'='*60}")
    
    return logger

logger = setup_pipeline_logger("INITIALIZATION")
logger.info(f"Working directory: {os.getcwd()}")
logger.info(f"Config loaded (embedded): top_k_models={config.top_k_models}, num_folds={config.num_folds}")

2026-05-20 05:42:07,700 - INFO - Working directory: /kaggle/working
2026-05-20 05:42:07,701 - INFO - Notebook: Pipe-1 Complete End-to-End Pipeline
2026-05-20 05:42:07,702 - INFO - Timestamp: 2026-05-20 05:42:07.702743


In [ ]:
# EXACT: Configuration from embedded config class (no external files)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device: {DEVICE}")

# Load all settings from config
IMAGENET_MEAN = config.imagenet_mean
IMAGENET_STD = config.imagenet_std
TARGET_SIZE = config.target_size
MODEL_NAME = config.model_name
PRETRAINED = config.pretrained
NUM_CLASSES = config.num_classes
EPOCHS = config.epochs
BATCH_SIZE_TRAIN = config.batch_size_train
BATCH_SIZE_VAL = config.batch_size_val
LEARNING_RATE = config.learning_rate
DROPOUT_RATE = config.dropout_rate
DROPOUT_PATH_RATE = config.dropout_path_rate
WEIGHT_DECAY = config.weight_decay
LABEL_SMOOTHING = config.label_smoothing
BATA = config.optimizer_betas
EPS = config.optimizer_eps
T_MAX = config.scheduler_t_max
NUM_FOLDS = config.num_folds
RANDOM_SEED = config.random_seed
TOP_K_MODELS = config.top_k_models
GRADIENT_CLIP_NORM = config.gradient_clip_norm

logger.info(f"""
Configuration Loaded (Embedded):
  Device: {DEVICE}
  Model: {MODEL_NAME} (pretrained={PRETRAINED})
  Epochs: {EPOCHS}
  Batch size: train={BATCH_SIZE_TRAIN}, val={BATCH_SIZE_VAL}
  Learning rate: {LEARNING_RATE}
  Target size: {TARGET_SIZE}x{TARGET_SIZE}
  Num folds: {NUM_FOLDS}
  Top K models to save: {TOP_K_MODELS}
""")

2026-05-20 05:42:07,724 - INFO - Device: cuda
2026-05-20 05:42:07,726 - INFO - 
Configuration:
  Device: cuda
  Model: tf_efficientnetv2_m.in21k_ft_in1k (pretrained=True)
  Epochs: 15
  Batch size: train=32, val=32
  Learning rate: 0.0001
  Target size: 224x224



---
# STAGE 1: Data Loading & Validation

In [ ]:
# STAGE 1: EXACT - Load CSVs (Kaggle structure)
logger = setup_pipeline_logger("DATA LOADING & VALIDATION")

BASE_INPUT = config.input_dir

train_df = pd.read_csv(f'{BASE_INPUT}/Data/training.csv')
test_df = pd.read_csv(f'{BASE_INPUT}/Data/test.csv')

logger.info(f"Loaded training.csv: {train_df.shape}")
logger.info(f"Loaded test.csv: {test_df.shape}")
logger.info(f"Train columns: {list(train_df.columns)}")
logger.info(f"Test columns: {list(test_df.columns)}")

# Verify column names
assert list(train_df.columns) == ['ID', 'path', 'y'], f"Got columns: {list(train_df.columns)}"
assert list(test_df.columns) == ['ID', 'path'], f"Got columns: {list(test_df.columns)}"

logger.info("✓ Column names verified")

2026-05-20 05:42:07,760 - INFO - 
2026-05-20 05:42:07,760 - INFO - STAGE 1: DATA LOADING & VALIDATION
2026-05-20 05:42:07,761 - INFO - ============================================================
2026-05-20 05:42:07,813 - INFO - Loaded training.csv: (7000, 3)
2026-05-20 05:42:07,813 - INFO - Loaded test.csv: (3000, 2)
2026-05-20 05:42:07,814 - INFO - Train columns: ['ID', 'path', 'y']
2026-05-20 05:42:07,815 - INFO - Test columns: ['ID', 'path']
2026-05-20 05:42:07,816 - INFO - ✓ Column names verified


In [6]:
# STAGE 1: EXACT - Verify file existence
logger.info("\nVerifying file existence...")

train_missing = []
for idx, row in train_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    if not os.path.exists(path):
        train_missing.append(path)

if train_missing:
    raise FileNotFoundError(f"Missing {len(train_missing)} training files: {train_missing[:5]}")
else:
    logger.info(f"✓ All {len(train_df)} training files exist")

test_missing = []
for idx, row in test_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    if not os.path.exists(path):
        test_missing.append(path)

if test_missing:
    raise FileNotFoundError(f"Missing {len(test_missing)} test files: {test_missing[:5]}")
else:
    logger.info(f"✓ All {len(test_df)} test files exist")

2026-05-20 05:42:07,838 - INFO - 
Verifying file existence...
2026-05-20 05:42:32,307 - INFO - ✓ All 7000 training files exist
2026-05-20 05:42:42,876 - INFO - ✓ All 3000 test files exist


In [7]:
# STAGE 1: EXACT - Extract image metadata
logger.info("\nExtracting image metadata...")

# Create class index to name mapping
class_idx_to_name = {
    0: 'AuraFlow',
    1: 'Freepik',
    2: 'Lumina',
    3: 'Photon',
    4: 'Pixart-sigma',
    5: 'Playground v2.5',
    6: 'StableDiffusion3',
    7: 'StableDiffusion3.5',
    8: 'StableDiffusionXL-Turbo',
    9: 'Tencent Hunyuan'
}

train_metadata = []
for idx, row in train_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    img = Image.open(path)
    size_bytes = os.path.getsize(path)
    train_metadata.append({
        'image_id': row['ID'],
        'source': class_idx_to_name[row['y']],
        'width': img.width,
        'height': img.height,
        'format': img.format,
        'size_kb': size_bytes / 1024,
        'mode': img.mode
    })
    if (idx + 1) % 1000 == 0:
        logger.info(f"Processed {idx + 1} training images")

train_metadata_df = pd.DataFrame(train_metadata)
logger.info(f"✓ Metadata extracted for {len(train_metadata_df)} images")

2026-05-20 05:42:42,906 - INFO - 
Extracting image metadata...
2026-05-20 05:42:59,583 - INFO - Processed 1000 training images
2026-05-20 05:43:14,919 - INFO - Processed 2000 training images
2026-05-20 05:43:27,215 - INFO - Processed 3000 training images
2026-05-20 05:43:38,940 - INFO - Processed 4000 training images
2026-05-20 05:43:51,748 - INFO - Processed 5000 training images
2026-05-20 05:44:04,024 - INFO - Processed 6000 training images
2026-05-20 05:44:16,127 - INFO - Processed 7000 training images
2026-05-20 05:44:16,139 - INFO - ✓ Metadata extracted for 7000 images


In [8]:
# STAGE 1: EXACT - Validate class distribution
logger.info("\nValidating class distribution...")

class_counts = train_metadata_df['source'].value_counts()
logger.info(f"\nClass distribution:\n{class_counts}")

# Verify each class has exactly 700 (7000 total / 10 classes)
expected_per_class = 700
for class_name, count in class_counts.items():
    assert count == expected_per_class, f"Class {class_name} has {count} images, expected {expected_per_class}"

logger.info(f"✓ All {len(class_counts)} classes have exactly {expected_per_class} images ({len(class_counts) * expected_per_class} total)")

# Verify no duplicate IDs
assert len(train_df) == len(train_df['ID'].unique()), "Duplicate image IDs found"
logger.info("✓ No duplicate image IDs")

2026-05-20 05:44:16,162 - INFO - 
Validating class distribution...
2026-05-20 05:44:16,172 - INFO - 
Class distribution:
source
AuraFlow                   700
StableDiffusionXL-Turbo    700
Playground v2.5            700
Tencent Hunyuan            700
StableDiffusion3.5         700
Freepik                    700
Pixart-sigma               700
StableDiffusion3           700
Photon                     700
Lumina                     700
Name: count, dtype: int64
2026-05-20 05:44:16,173 - INFO - ✓ All 10 classes have exactly 700 images (7000 total)
2026-05-20 05:44:16,177 - INFO - ✓ No duplicate image IDs


---
# STAGE 2: Exploratory Data Analysis (EDA)

In [ ]:
# STAGE 2: EXACT - Setup EDA outputs
logger = setup_pipeline_logger("EXPLORATORY DATA ANALYSIS (EDA)")

config.get_eda_dir().mkdir(parents=True, exist_ok=True)
(config.get_eda_dir() / "plots").mkdir(parents=True, exist_ok=True)
logger.info(f"EDA output directory: {config.get_eda_dir()}")

2026-05-20 05:44:16,212 - INFO - 
2026-05-20 05:44:16,213 - INFO - STAGE 2: EXPLORATORY DATA ANALYSIS (EDA)
2026-05-20 05:44:16,214 - INFO - ============================================================
2026-05-20 05:44:16,215 - INFO - Created eda_outputs directory


In [ ]:
# STAGE 2: EXACT - Class distribution analysis
logger.info("\nClass distribution analysis...")

fig, ax = plt.subplots(figsize=(12, 6))
class_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Generator Class')
ax.set_ylabel('Number of Images')
ax.set_title('Training Data: Class Distribution')
ax.set_ylim([900, 1100])
for i, v in enumerate(class_counts):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(config.get_eda_dir() / 'class_distribution.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved class_distribution.png")

2026-05-20 05:44:16,237 - INFO - 
Class distribution analysis...
2026-05-20 05:44:16,585 - INFO - ✓ Saved class_distribution.png


In [ ]:
# STAGE 2: EXACT - Image dimension analysis
logger.info("Image dimension analysis...")

logger.info(f"Width:  mean={train_metadata_df['width'].mean():.1f}, std={train_metadata_df['width'].std():.1f}")
logger.info(f"        min={train_metadata_df['width'].min()}, max={train_metadata_df['width'].max()}")
logger.info(f"Height: mean={train_metadata_df['height'].mean():.1f}, std={train_metadata_df['height'].std():.1f}")
logger.info(f"        min={train_metadata_df['height'].min()}, max={train_metadata_df['height'].max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_metadata_df['width'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Width Distribution')
axes[1].hist(train_metadata_df['height'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Height Distribution')
plt.tight_layout()
plt.savefig(config.get_eda_dir() / 'plots' / 'dimension_distribution.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved dimension_distribution.png")

2026-05-20 05:44:16,609 - INFO - Image dimension analysis...
2026-05-20 05:44:16,611 - INFO - Width:  mean=921.6, std=204.8
2026-05-20 05:44:16,612 - INFO -         min=512, max=1024
2026-05-20 05:44:16,614 - INFO - Height: mean=921.6, std=204.8
2026-05-20 05:44:16,615 - INFO -         min=512, max=1024
2026-05-20 05:44:16,991 - INFO - ✓ Saved dimension_distribution.png


In [ ]:
# STAGE 2: EXACT - File format and size analysis
logger.info("File format and size analysis...")

format_counts = train_metadata_df['format'].value_counts()
logger.info(f"Image formats:\n{format_counts}")

logger.info(f"Image size (KB) - Mean: {train_metadata_df['size_kb'].mean():.2f}")
logger.info(f"                  Std:  {train_metadata_df['size_kb'].std():.2f}")
logger.info(f"                  Min:  {train_metadata_df['size_kb'].min():.2f}")
logger.info(f"                  Max:  {train_metadata_df['size_kb'].max():.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([train_metadata_df[train_metadata_df['source'] == src]['size_kb'].values 
            for src in sorted(train_metadata_df['source'].unique())],
           labels=sorted(train_metadata_df['source'].unique()))
ax.set_ylabel('File Size (KB)')
ax.set_title('File Size Distribution by Generator')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(config.get_eda_dir() / 'plots' / 'filesize_boxplot.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved filesize_boxplot.png")

2026-05-20 05:44:17,016 - INFO - File format and size analysis...
2026-05-20 05:44:17,018 - INFO - Image formats:
format
PNG    7000
Name: count, dtype: int64
2026-05-20 05:44:17,020 - INFO - Image size (KB) - Mean: 1139.01
2026-05-20 05:44:17,021 - INFO -                   Std:  425.50
2026-05-20 05:44:17,022 - INFO -                   Min:  200.76
2026-05-20 05:44:17,023 - INFO -                   Max:  2212.52


/tmp/ipykernel_22/3503238709.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([train_metadata_df[train_metadata_df['source'] == src]['size_kb'].values


2026-05-20 05:44:17,295 - INFO - ✓ Saved filesize_boxplot.png


In [ ]:
# STAGE 2: EXACT - Generator sample grid
logger.info("Generator sample grid visualization...")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

classes = sorted(train_metadata_df['source'].unique())
for col_idx, class_name in enumerate(classes):
    class_images = train_metadata_df[train_metadata_df['source'] == class_name]
    sample_row = class_images.sample(1).iloc[0]
    
    # Find matching row in train_df to get path
    train_row = train_df[train_df['ID'] == sample_row['image_id']].iloc[0]
    img_path = f"{BASE_INPUT}{train_row['path']}"
    img = Image.open(img_path)
    
    axes[col_idx].imshow(img)
    axes[col_idx].set_title(class_name, fontsize=10, fontweight='bold')
    axes[col_idx].axis('off')

plt.tight_layout()
plt.savefig(config.get_eda_dir() / 'plots' / 'generator_samples.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved generator_samples.png")

2026-05-20 05:44:17,321 - INFO - Generator sample grid visualization...
2026-05-20 05:44:19,459 - INFO - ✓ Saved generator_samples.png


In [ ]:
# STAGE 2: EXACT - Save EDA insights
logger.info("Saving EDA insights...")

eda_insights = {
    'class_distribution': class_counts.to_dict(),
    'image_dimensions': {
        'width': {
            'mean': float(train_metadata_df['width'].mean()),
            'std': float(train_metadata_df['width'].std()),
            'min': int(train_metadata_df['width'].min()),
            'max': int(train_metadata_df['width'].max())
        },
        'height': {
            'mean': float(train_metadata_df['height'].mean()),
            'std': float(train_metadata_df['height'].std()),
            'min': int(train_metadata_df['height'].min()),
            'max': int(train_metadata_df['height'].max())
        }
    },
    'file_format': format_counts.to_dict(),
    'file_size_kb': {
        'mean': float(train_metadata_df['size_kb'].mean()),
        'std': float(train_metadata_df['size_kb'].std()),
        'min': float(train_metadata_df['size_kb'].min()),
        'max': float(train_metadata_df['size_kb'].max())
    },
    'image_modes': train_metadata_df['mode'].value_counts().to_dict(),
    'num_training_images': len(train_df),
    'num_test_images': len(test_df),
    'num_classes': len(class_counts)
}

with open(config.get_eda_dir() / 'eda_insights.json', 'w') as f:
    json.dump(eda_insights, f, indent=2)

logger.info("✓ Saved eda_insights.json")
logger.info("\n✓ STAGE 2 COMPLETE: EDA finished")

2026-05-20 05:44:19,488 - INFO - Saving EDA insights...
2026-05-20 05:44:19,491 - INFO - ✓ Saved eda_insights.json
2026-05-20 05:44:19,492 - INFO - 
✓ STAGE 2 COMPLETE: EDA finished


---
# STAGE 3: Preprocessing

In [15]:
# STAGE 3: EXACT - Define preprocessing function
logger.info("\n" + "="*60)
logger.info("STAGE 3: PREPROCESSING")
logger.info("="*60)

logger.info(f"Preprocessing Configuration:")
logger.info(f"  Target size: {TARGET_SIZE}x{TARGET_SIZE}")
logger.info(f"  Normalization: ImageNet mean={IMAGENET_MEAN}, std={IMAGENET_STD}")

def preprocess_image(image_path, target_size=224):
    """
    Load, resize, and normalize a single image.
    """
    try:
        # Load image
        img = Image.open(image_path)
        
        # Convert RGBA → RGB
        if img.mode == 'RGBA':
            rgb_img = Image.new('RGB', img.size, (255, 255, 255))
            rgb_img.paste(img, mask=img.split()[3])
            img = rgb_img
        elif img.mode == 'L':
            img = img.convert('RGB')
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize
        img_resized = img.resize((target_size, target_size), Image.LANCZOS)
        
        # Convert to numpy array
        img_array = np.array(img_resized, dtype=np.float32)
        
        # Normalize to [0, 1]
        img_array = img_array / 255.0
        
        # ImageNet normalization
        img_array = (img_array - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
        
        return img_array
        
    except Exception as e:
        logger.error(f"Error processing {image_path}: {e}")
        raise

logger.info("✓ Preprocessing function defined")

2026-05-20 05:44:19,537 - INFO - 
2026-05-20 05:44:19,538 - INFO - STAGE 3: PREPROCESSING
2026-05-20 05:44:19,539 - INFO - ============================================================
2026-05-20 05:44:19,540 - INFO - Preprocessing Configuration:
2026-05-20 05:44:19,541 - INFO -   Target size: 224x224
2026-05-20 05:44:19,542 - INFO -   Normalization: ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
2026-05-20 05:44:19,543 - INFO - ✓ Preprocessing function defined


In [16]:
# STAGE 3: EXACT - Preprocess all training images
logger.info(f"\nPreprocessing {len(train_df)} training images...")

X_train = np.zeros((len(train_df), 224, 224, 3), dtype=np.float32)

for idx, row in train_df.iterrows():
    image_path = f"{BASE_INPUT}{row['path']}"
    X_train[idx] = preprocess_image(image_path, target_size=224)
    
    if (idx + 1) % 1000 == 0:
        logger.info(f"  Processed {idx + 1} training images")

logger.info(f"✓ Loaded all training images: X_train.shape = {X_train.shape}")
logger.info(f"  Value range: min={X_train.min():.3f}, max={X_train.max():.3f}")

assert X_train.min() < -1.0, "Preprocessing may have failed (values too high)"
assert X_train.max() > 1.0, "Preprocessing may have failed (values too low)"
assert not np.isnan(X_train).any(), "Found NaN values in X_train!"
assert not np.isinf(X_train).any(), "Found Inf values in X_train!"
logger.info(f"✓ Validation checks passed")

2026-05-20 05:44:19,567 - INFO - 
Preprocessing 7000 training images...
2026-05-20 05:45:05,568 - INFO -   Processed 1000 training images
2026-05-20 05:45:51,664 - INFO -   Processed 2000 training images
2026-05-20 05:46:37,272 - INFO -   Processed 3000 training images
2026-05-20 05:47:23,154 - INFO -   Processed 4000 training images
2026-05-20 05:48:09,929 - INFO -   Processed 5000 training images
2026-05-20 05:48:56,930 - INFO -   Processed 6000 training images
2026-05-20 05:49:43,530 - INFO -   Processed 7000 training images
2026-05-20 05:49:43,531 - INFO - ✓ Loaded all training images: X_train.shape = (7000, 224, 224, 3)
2026-05-20 05:49:44,264 - INFO -   Value range: min=-2.118, max=2.640
2026-05-20 05:49:46,351 - INFO - ✓ Validation checks passed


In [17]:
# STAGE 3: EXACT - Preprocess all test images
logger.info(f"\nPreprocessing {len(test_df)} test images...")

X_test = np.zeros((len(test_df), 224, 224, 3), dtype=np.float32)

for idx, row in test_df.iterrows():
    image_path = f"{BASE_INPUT}{row['path']}"
    X_test[idx] = preprocess_image(image_path, target_size=224)
    
    if (idx + 1) % 500 == 0:
        logger.info(f"  Processed {idx + 1} test images")

logger.info(f"✓ Loaded all test images: X_test.shape = {X_test.shape}")
logger.info(f"  Value range: min={X_test.min():.3f}, max={X_test.max():.3f}")

assert not np.isnan(X_test).any(), "Found NaN values in X_test!"
assert not np.isinf(X_test).any(), "Found Inf values in X_test!"
logger.info(f"✓ Validation checks passed")

2026-05-20 05:49:46,380 - INFO - 
Preprocessing 3000 test images...
2026-05-20 05:50:18,351 - INFO -   Processed 500 test images
2026-05-20 05:50:53,242 - INFO -   Processed 1000 test images
2026-05-20 05:51:26,021 - INFO -   Processed 1500 test images
2026-05-20 05:51:57,781 - INFO -   Processed 2000 test images
2026-05-20 05:52:34,205 - INFO -   Processed 2500 test images
2026-05-20 05:53:06,545 - INFO -   Processed 3000 test images
2026-05-20 05:53:06,546 - INFO - ✓ Loaded all test images: X_test.shape = (3000, 224, 224, 3)
2026-05-20 05:53:06,863 - INFO -   Value range: min=-2.118, max=2.640
2026-05-20 05:53:07,454 - INFO - ✓ Validation checks passed


In [ ]:
# STAGE 3: EXACT - Save preprocessed data
logger.info("\nSaving preprocessed data...")

preprocessed_dir = config.resolve_path('preprocessed')
preprocessed_dir.mkdir(parents=True, exist_ok=True)

np.save(preprocessed_dir / 'X_train_preprocessed.npy', X_train)
np.save(preprocessed_dir / 'X_test_preprocessed.npy', X_test)

logger.info(f"✓ Saved X_train_preprocessed.npy ({X_train.nbytes / 1e9:.2f} GB)")
logger.info(f"✓ Saved X_test_preprocessed.npy ({X_test.nbytes / 1e9:.2f} GB)")

preprocessing_metadata = {
    'target_size': 224,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'X_train_shape': list(X_train.shape),
    'X_test_shape': list(X_test.shape),
    'X_train_value_range': [float(X_train.min()), float(X_train.max())],
    'X_test_value_range': [float(X_test.min()), float(X_test.max())],
    'timestamp': str(pd.Timestamp.now())
}

with open(preprocessed_dir / 'normalization_metadata.json', 'w') as f:
    json.dump(preprocessing_metadata, f, indent=2)

logger.info(f"✓ Saved normalization_metadata.json")
logger.info("\n✓ STAGE 3 COMPLETE: Preprocessing finished")

2026-05-20 05:53:07,482 - INFO - 
Saving preprocessed data...
2026-05-20 05:53:13,777 - INFO - ✓ Saved X_train_preprocessed.npy (4.21 GB)
2026-05-20 05:53:13,778 - INFO - ✓ Saved X_test_preprocessed.npy (1.81 GB)
2026-05-20 05:53:14,853 - INFO - ✓ Saved normalization_metadata.json
2026-05-20 05:53:14,854 - INFO - 
✓ STAGE 3 COMPLETE: Preprocessing finished


---
# STAGE 4: Train/Validation Stratified Split

In [ ]:
# STAGE 5: EXACT - Define augmentation pipelines
logger = setup_pipeline_logger("MODEL TRAINING")

train_augmentation = A.Compose([
    A.Rotate(limit=config.augmentation_rotate, p=0.7),
    A.HorizontalFlip(p=config.augmentation_hflip_prob),
    A.VerticalFlip(p=config.augmentation_vflip_prob),
    A.RandomBrightnessContrast(
        brightness_limit=config.augmentation_brightness,
        contrast_limit=config.augmentation_contrast,
        p=config.augmentation_brightness_contrast_prob
    ),
    A.GaussianBlur(blur_limit=(3, 3), p=config.augmentation_blur_prob),
    A.RandomCrop(config.target_size, config.target_size, p=config.augmentation_crop_prob),
    A.Normalize(
        mean=config.imagenet_mean,
        std=config.imagenet_std,
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

val_augmentation = A.Compose([
    A.Normalize(
        mean=config.imagenet_mean,
        std=config.imagenet_std,
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

logger.info("✓ Augmentation pipelines defined")

2026-05-20 05:53:14,903 - INFO - 
2026-05-20 05:53:14,904 - INFO - STAGE 4: TRAIN/VALIDATION STRATIFIED SPLIT
2026-05-20 05:53:14,905 - INFO - ============================================================
2026-05-20 05:53:14,906 - INFO - Creating 5-Fold Stratified Cross-Validation...
2026-05-20 05:53:14,907 - INFO -   n_splits: 5
2026-05-20 05:53:14,907 - INFO -   shuffle: True
2026-05-20 05:53:14,908 - INFO -   random_state: 42
2026-05-20 05:53:14,920 - INFO - Created 5 folds with indices saved


In [20]:
# STAGE 4: EXACT - Verify fold balance
logger.info("\nVerifying fold balance...")

for fold_num in range(5):
    fold_key = f'fold_{fold_num}'
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    
    val_labels = y_train[val_indices]
    class_counts_fold = np.bincount(val_labels, minlength=10)
    
    assert len(class_counts_fold) == 10, f"Fold {fold_num} missing classes!"
    
    logger.info(f"Fold {fold_num}: ~{class_counts_fold.mean():.0f} images per class")
    
    for class_idx, count in enumerate(class_counts_fold):
        ratio = count / class_counts_fold.mean()
        assert 0.8 < ratio < 1.2, f"Fold {fold_num} class {class_idx} imbalanced ({ratio:.2f}x)"

logger.info("✓ All folds have balanced class distribution")

2026-05-20 05:53:14,947 - INFO - 
Verifying fold balance...
2026-05-20 05:53:14,949 - INFO - Fold 0: ~140 images per class
2026-05-20 05:53:14,950 - INFO - Fold 1: ~140 images per class
2026-05-20 05:53:14,951 - INFO - Fold 2: ~140 images per class
2026-05-20 05:53:14,952 - INFO - Fold 3: ~140 images per class
2026-05-20 05:53:14,953 - INFO - Fold 4: ~140 images per class
2026-05-20 05:53:14,955 - INFO - ✓ All folds have balanced class distribution


In [ ]:
# STAGE 4: EXACT - Save fold metadata
logger.info("\nSaving fold metadata...")

preprocessed_dir = config.resolve_path('preprocessed')
preprocessed_dir.mkdir(parents=True, exist_ok=True)

with open(preprocessed_dir / 'fold_metadata.json', 'w') as f:
    json.dump(fold_metadata, f, indent=2)

logger.info("✓ Saved fold_metadata.json")
logger.info("\n✓ STAGE 4 COMPLETE: CV splits created")

2026-05-20 05:53:14,982 - INFO - 
Saving fold metadata...
2026-05-20 05:53:14,997 - INFO - ✓ Saved fold_metadata.json
2026-05-20 05:53:14,998 - INFO - 
✓ STAGE 4 COMPLETE: CV splits created


---
# STAGE 5: Model Training

In [22]:
# STAGE 5: EXACT - Define augmentation pipelines
logger.info("\n" + "="*60)
logger.info("STAGE 5: MODEL TRAINING")
logger.info("="*60)

train_augmentation = A.Compose([
    A.Rotate(limit=5, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(
        brightness_limit=(-0.2, 0.2),
        contrast_limit=(-0.2, 0.2),
        p=0.5
    ),
    A.GaussianBlur(blur_limit=(3, 3), p=0.3),
    A.RandomCrop(224, 224, p=0.1),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

val_augmentation = A.Compose([
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

logger.info("✓ Augmentation pipelines defined")

2026-05-20 05:53:15,051 - INFO - 
2026-05-20 05:53:15,052 - INFO - STAGE 5: MODEL TRAINING
2026-05-20 05:53:15,053 - INFO - ============================================================
2026-05-20 05:53:15,071 - INFO - ✓ Augmentation pipelines defined


In [23]:
# STAGE 5: EXACT - Define Dataset class
class ImageDataset(Dataset):
    def __init__(self, images, labels, augmentation=None):
        self.images = images
        self.labels = labels
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.augmentation is not None:
            augmented = self.augmentation(image=image)
            image = augmented['image']
        
        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long)
        }

logger.info("✓ ImageDataset class defined")

2026-05-20 05:53:15,100 - INFO - ✓ ImageDataset class defined


In [ ]:
# STAGE 5: EXACT - Training loop (ALL 5 FOLDS)
logger.info("\nStarting training loop...")

config.get_logs_dir().mkdir(parents=True, exist_ok=True)
csv_file = str(config.resolve_path(f"{config.logs}/training_log.csv"))
csv_header = ['fold', 'epoch', 'train_loss', 'train_acc', 'train_f1', 
              'val_loss', 'val_acc', 'val_f1', 'learning_rate', 'time_sec']

with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(csv_header)

logger.info(f"Training log will be saved to {csv_file}")

# Loop through folds
for fold_num in range(NUM_FOLDS):
    logger.info(f"\n{'='*60}")
    logger.info(f"FOLD {fold_num}/{NUM_FOLDS}")
    logger.info(f"{'='*60}")
    
    # Load fold indices
    fold_key = f'fold_{fold_num}'
    train_indices = np.array(fold_metadata[fold_key]['train_indices'])
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    
    logger.info(f"Train: {len(train_indices)} | Val: {len(val_indices)}")
    
    # Get train/val labels (already numeric 0-9)
    y_train_fold_idx = train_df.iloc[train_indices]['y'].values
    y_val_fold_idx = train_df.iloc[val_indices]['y'].values
    
    # Get train/val images
    X_train_fold = X_train[train_indices]
    X_val_fold = X_train[val_indices]
    
    # Create datasets
    train_dataset = ImageDataset(X_train_fold, y_train_fold_idx, augmentation=train_augmentation)
    val_dataset = ImageDataset(X_val_fold, y_val_fold_idx, augmentation=val_augmentation)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE_VAL,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    logger.info(f"DataLoaders created: {len(train_loader)} train batches, {len(val_loader)} val batches")
    
    # Initialize model
    model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE, drop_path_rate=DROPOUT_PATH_RATE)
    model = model.to(DEVICE)
    
    # Loss function
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=BATA,
        eps=EPS,
        weight_decay=WEIGHT_DECAY
    )
    
    # Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=config.scheduler_eta_min)
    
    logger.info(f"Model initialized: {MODEL_NAME}")
    logger.info(f"Optimizer: AdamW (lr={LEARNING_RATE})")
    logger.info(f"Scheduler: CosineAnnealingLR")
    
    # Create checkpoint directory
    checkpoint_dir = config.get_checkpoint_dir(fold_num)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    # Training loop
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        
        # TRAIN PHASE
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for batch_idx, batch in enumerate(train_loader):
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_NORM)
            optimizer.step()
            
            train_loss += loss.item()
            train_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='macro', zero_division=0)
        
        # VAL PHASE
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_loader):
                images = batch['image'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
                val_labels.extend(labels.detach().cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        
        # Update scheduler
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        epoch_time = time.time() - epoch_start
        
        # Log epoch
        logger.info(f"Fold {fold_num} | Epoch {epoch+1:3d}/{EPOCHS} | "
                   f"train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | "
                   f"train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f} | "
                   f"train_f1: {train_f1:.4f} | val_f1: {val_f1:.4f} | "
                   f"lr: {current_lr:.2e} | time: {epoch_time:.1f}s")
        
        # Save checkpoint
        checkpoint_path = checkpoint_dir / f"epoch_{epoch:03d}.pth"
        torch.save({
            'epoch': epoch,
            'fold': fold_num,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1': train_f1,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'val_f1': val_f1,
            'learning_rate': current_lr,
            'model_name': MODEL_NAME
        }, checkpoint_path)
        
        # Write to CSV
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([fold_num, epoch, train_loss, train_acc, train_f1,
                           val_loss, val_acc, val_f1, current_lr, epoch_time])
    
    logger.info(f"✓ Fold {fold_num} training complete. Saved {EPOCHS} checkpoints to {checkpoint_dir}/")
    
    # Free memory
    del model, optimizer, scheduler
    del train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()

logger.info(f"\n✓ Training complete for all {NUM_FOLDS} folds")
logger.info(f"✓ Total checkpoints saved: {NUM_FOLDS * EPOCHS} ({NUM_FOLDS} folds × {EPOCHS} epochs)")
logger.info(f"✓ Training log: {csv_file}")

2026-05-20 05:53:15,142 - INFO - 
Starting training loop...
2026-05-20 05:53:15,144 - INFO - Training log will be saved to /kaggle/working/logs/training_log.csv
2026-05-20 05:53:15,146 - INFO - 
2026-05-20 05:53:15,146 - INFO - FOLD 0
2026-05-20 05:53:15,148 - INFO - ============================================================
2026-05-20 05:53:15,149 - INFO - Train: 5600 | Val: 1400
2026-05-20 05:53:19,814 - INFO - DataLoaders created: 175 train batches, 44 val batches
2026-05-20 05:53:20,771 - INFO - Loading pretrained weights from Hugging Face hub (timm/tf_efficientnetv2_m.in21k_ft_in1k)
2026-05-20 05:53:20,952 - INFO - HTTP Request: HEAD https://huggingface.co/timm/tf_efficientnetv2_m.in21k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-05-20 05:53:20,954 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-20 05:53:21,098 - INFO - HTTP Request: GET https://huggingface.co/api/models/timm/tf_efficientnetv2_m.in21k_ft_in1k/xet-read-token/631698f40fee81ab20478c0724337f1b21c06b28 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

2026-05-20 05:53:26,256 - INFO - [timm/tf_efficientnetv2_m.in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-20 05:53:26,384 - INFO - Missing keys (classifier.weight, classifier.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
2026-05-20 05:53:26,908 - INFO - Model initialized: tf_efficientnetv2_m.in21k_ft_in1k
2026-05-20 05:53:26,909 - INFO - Optimizer: AdamW (lr=0.0001)
2026-05-20 05:53:26,909 - INFO - Scheduler: CosineAnnealingLR
2026-05-20 05:55:27,193 - INFO - Fold 0 | Epoch   1/15 | train_loss: 3.7663 | val_loss: 1.1361 | train_acc: 0.2770 | val_acc: 0.7386 | train_f1: 0.2768 | val_f1: 0.7357 | lr: 9.89e-05 | time: 120.3s
2026-05-20 05:57:31,113 - INFO - Fold 0 | Epoch   2/15 | train_loss: 1.4873 | val_loss: 0.8882 | train_acc: 0.6029 | val_acc: 0.8664 | train_f1: 0.6024 | val_f1: 0.8630 | lr: 9.57e-05 | time: 123.0s
2026-05-20 05:59:35,037 - INFO - F

---
# STAGE 6: Validation

In [ ]:
# STAGE 6: EXACT - Evaluate all checkpoints
logger = setup_pipeline_logger("VALIDATION")

val_augmentation = A.Compose([
    A.Normalize(
        mean=config.imagenet_mean,
        std=config.imagenet_std,
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

validation_results = []

for fold_num in range(NUM_FOLDS):
    logger.info(f"\n{'='*60}")
    logger.info(f"VALIDATING FOLD {fold_num}/{NUM_FOLDS}")
    logger.info(f"{'='*60}")
    
    # Load fold indices and data
    fold_key = f'fold_{fold_num}'
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    X_val_fold = X_train[val_indices]
    y_val_fold_idx = train_df.iloc[val_indices]['y'].values
    
    # Create dataset and loader
    val_dataset = ImageDataset(X_val_fold, y_val_fold_idx, augmentation=val_augmentation)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_VAL, shuffle=False, num_workers=2)
    
    checkpoint_dir = config.get_checkpoint_dir(fold_num)
    
    # Evaluate all EPOCHS checkpoints for this fold
    for epoch in range(EPOCHS):
        checkpoint_path = checkpoint_dir / f"epoch_{epoch:03d}.pth"
        
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        
        # Load model
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(DEVICE)
        model.eval()
        
        # Evaluate
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                
                outputs = model(images)
                val_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
                val_labels.extend(labels.detach().cpu().numpy())
        
        # Compute metrics
        accuracy = accuracy_score(val_labels, val_preds)
        f1_macro = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        
        # Get train metrics from checkpoint
        train_acc = checkpoint['train_acc']
        train_f1 = checkpoint['train_f1']
        
        validation_results.append({
            'fold': fold_num,
            'epoch': epoch,
            'train_acc': train_acc,
            'train_f1': train_f1,
            'val_acc': accuracy,
            'val_f1': f1_macro,
            'checkpoint_path': str(checkpoint_path)
        })
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            logger.info(f"  Epoch {epoch:3d} | train_acc: {train_acc:.4f} | val_acc: {accuracy:.4f}")
    
    logger.info(f"✓ Validated all {EPOCHS} checkpoints for fold {fold_num}")
    del model
    torch.cuda.empty_cache()

logger.info(f"\n✓ Validation complete for all {NUM_FOLDS * EPOCHS} checkpoints")

# Convert to DataFrame
val_df = pd.DataFrame(validation_results)
validation_csv = str(config.resolve_path(f"{config.logs}/validation_metrics.csv"))
val_df.to_csv(validation_csv, index=False)

logger.info(f"✓ Saved validation_metrics.csv ({len(val_df)} rows)")

2026-05-20 06:24:25,854 - INFO - 
2026-05-20 06:24:25,856 - INFO - STAGE 6: VALIDATION
2026-05-20 06:24:25,856 - INFO - ============================================================
2026-05-20 06:24:25,860 - INFO - 
2026-05-20 06:24:25,861 - INFO - VALIDATING FOLD 0
2026-05-20 06:24:25,861 - INFO - ============================================================
2026-05-20 06:27:27,703 - INFO - ✓ Validated all 100 checkpoints for fold 0
2026-05-20 06:27:27,746 - INFO - 
✓ Validation complete for all 500 checkpoints
2026-05-20 06:27:27,790 - INFO - ✓ Saved validation_metrics.csv (15 rows)


---
# STAGE 7: Checkpoint Selection

In [ ]:
# STAGE 7: EXACT - Compute Generalization Score
logger = setup_pipeline_logger("CHECKPOINT SELECTION - TOP 5 MODELS")

val_df['gen_score'] = val_df['val_acc'] - np.abs(val_df['train_acc'] - val_df['val_acc'])

logger.info(f"Generalization Score computed for all checkpoints")
logger.info(f"  Gen_Score range: [{val_df['gen_score'].min():.4f}, {val_df['gen_score'].max():.4f}]")
logger.info(f"  Formula: gen_score = val_acc - |train_acc - val_acc|")
logger.info(f"  (Penalizes overfitting, rewards good validation accuracy)")

2026-05-20 06:27:27,845 - INFO - 
2026-05-20 06:27:27,846 - INFO - STAGE 7: CHECKPOINT SELECTION
2026-05-20 06:27:27,847 - INFO - ============================================================
2026-05-20 06:27:27,860 - INFO - Generalization Score computed for all checkpoints
2026-05-20 06:27:27,866 - INFO -   Gen_Score range: [0.2770, 0.9680]


In [ ]:
# STAGE 7: EXACT - Select TOP K best checkpoints across ALL folds
logger.info(f"\nSelecting top {TOP_K_MODELS} checkpoints across all {NUM_FOLDS} folds...")

# Sort by generalization score
val_df_sorted = val_df.sort_values('gen_score', ascending=False)

# Select top K
top_k_checkpoints = val_df_sorted.head(TOP_K_MODELS).copy()
top_k_checkpoints = top_k_checkpoints.reset_index(drop=True)

logger.info(f"\nTop {TOP_K_MODELS} checkpoints:")
for idx, row in top_k_checkpoints.iterrows():
    rank = idx + 1
    logger.info(f"  #{rank}: Fold {row['fold']} Epoch {row['epoch']:2d} | "
               f"Gen_Score={row['gen_score']:.4f} | Val_Acc={row['val_acc']:.4f} | "
               f"Train_Acc={row['train_acc']:.4f}")

best_checkpoints = {}
for idx, row in top_k_checkpoints.iterrows():
    best_checkpoints[idx] = {
        'fold': int(row['fold']),
        'epoch': int(row['epoch']),
        'gen_score': float(row['gen_score']),
        'train_acc': float(row['train_acc']),
        'val_acc': float(row['val_acc']),
        'checkpoint_path': row['checkpoint_path']
    }

2026-05-20 06:27:27,898 - INFO - 
Selecting best checkpoint per fold...
2026-05-20 06:27:27,905 - INFO - Fold 0: Selected epoch  12 (Gen_Score=0.9680, train_acc=0.9680, val_acc=0.9764)


ValueError: attempt to get argmax of an empty sequence

In [ ]:
# STAGE 7: EXACT - Copy only TOP K best checkpoints to final_models
logger.info(f"\nCopying top {TOP_K_MODELS} checkpoints to final_models...")

config.get_final_models_dir().mkdir(parents=True, exist_ok=True)

for model_idx in range(TOP_K_MODELS):
    best_info = best_checkpoints[model_idx]
    source_path = best_info['checkpoint_path']
    dest_name = f"model_rank_{model_idx+1:02d}_fold{best_info['fold']}_ep{best_info['epoch']:02d}.pth"
    dest_path = config.get_final_models_dir() / dest_name
    
    shutil.copy(source_path, dest_path)
    logger.info(f"  #{model_idx+1}: {dest_name} (gen_score={best_info['gen_score']:.4f})")

logger.info(f"✓ Copied {TOP_K_MODELS} top checkpoints")

In [ ]:
# STAGE 7: EXACT - Save selection rationale and cleanup
logger.info("\nSaving selection rationale and cleaning up old checkpoints...")

selection_rationale = {
    'formula': 'Gen_Score = val_acc - |train_acc - val_acc|',
    'description': 'Selects checkpoints that generalize well (penalizes overfitting)',
    'strategy': f'Top {TOP_K_MODELS} across all {NUM_FOLDS} folds',
    'selected_models': {}
}

for model_idx in range(TOP_K_MODELS):
    best_info = best_checkpoints[model_idx]
    selection_rationale['selected_models'][f'rank_{model_idx+1}'] = best_info

rationale_path = config.get_final_models_dir() / "checkpoint_selection_rationale.json"
with open(rationale_path, 'w') as f:
    json.dump(selection_rationale, f, indent=2)

logger.info("✓ Saved checkpoint_selection_rationale.json")

# Also save as CSV
selection_df = pd.DataFrame([
    {
        'rank': r+1,
        'fold': best_checkpoints[r]['fold'],
        'epoch': best_checkpoints[r]['epoch'],
        'gen_score': best_checkpoints[r]['gen_score'],
        'train_acc': best_checkpoints[r]['train_acc'],
        'val_acc': best_checkpoints[r]['val_acc']
    }
    for r in range(TOP_K_MODELS)
])
selection_csv = config.get_final_models_dir() / "checkpoint_selection_summary.csv"
selection_df.to_csv(selection_csv, index=False)

logger.info("✓ Saved checkpoint_selection_summary.csv")

# CLEANUP: Delete all other checkpoints to save space
logger.info(f"\nCleaning up old checkpoints (keeping only top {TOP_K_MODELS})...")
checkpoint_base_dir = config.resolve_path(config.checkpoint_base)
total_deleted = 0

for fold_num in range(NUM_FOLDS):
    fold_dir = config.get_checkpoint_dir(fold_num)
    if fold_dir.exists():
        for checkpoint_file in fold_dir.glob("*.pth"):
            # Check if this checkpoint is in the top K
            is_kept = False
            for model_idx in range(TOP_K_MODELS):
                if best_checkpoints[model_idx]['checkpoint_path'] == str(checkpoint_file):
                    is_kept = True
                    break
            
            if not is_kept:
                checkpoint_file.unlink()
                total_deleted += 1

logger.info(f"✓ Deleted {total_deleted} old checkpoints, keeping only {TOP_K_MODELS}")
logger.info(f"  Estimated space saved: {(total_deleted * 110) / 1024:.1f} GB")

In [ ]:
# STAGE 7: EXACT - Verify selected checkpoints
logger.info(f"\nVerifying {TOP_K_MODELS} selected checkpoints...")

for model_idx in range(TOP_K_MODELS):
    model_name = f"model_rank_{model_idx+1:02d}_fold{best_checkpoints[model_idx]['fold']}_ep{best_checkpoints[model_idx]['epoch']:02d}.pth"
    dest_path = config.get_final_models_dir() / model_name
    
    # Check file exists
    assert dest_path.exists(), f"Missing {dest_path}"
    
    # Check file size
    file_size_mb = dest_path.stat().st_size / 1e6
    assert 90 < file_size_mb < 150, f"Checkpoint size unexpected: {file_size_mb:.0f} MB"
    
    # Load and verify checkpoint structure
    checkpoint = torch.load(dest_path, map_location='cpu')
    assert 'model_state_dict' in checkpoint, "Missing model_state_dict"
    assert 'train_acc' in checkpoint, "Missing train_acc"
    assert 'val_acc' in checkpoint, "Missing val_acc"
    
    logger.info(f"✓ {model_name} verified")

logger.info(f"\n✓ All {TOP_K_MODELS} checkpoints validated successfully")
logger.info(f"\n✓ STAGE 7 COMPLETE: Checkpoint selection finished")

---
# STAGE 8: Inference & Submission

In [ ]:
# STAGE 8: EXACT - Load test data
logger.info("\n" + "="*60)
logger.info("STAGE 8: INFERENCE & SUBMISSION")
logger.info("="*60)

class ImageDatasetTest(Dataset):
    def __init__(self, images, augmentation=None):
        self.images = images
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        
        if self.augmentation is not None:
            augmented = self.augmentation(image=image)
            image = augmented['image']
        
        return image

val_augmentation = A.Compose([
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

logger.info("Loading test images...")
test_dataset = ImageDatasetTest(X_test, augmentation=val_augmentation)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_VAL,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

logger.info(f"✓ Test loader created: {len(test_loader)} batches")

In [ ]:
# STAGE 8: EXACT - Generate predictions from top K models
logger = setup_pipeline_logger("INFERENCE & SUBMISSION")

logger.info(f"\nGenerating predictions from top {TOP_K_MODELS} models...")

test_predictions_all_models = []
model_scores = []

for model_idx in range(TOP_K_MODELS):
    model_name = f"model_rank_{model_idx+1:02d}_fold{best_checkpoints[model_idx]['fold']}_ep{best_checkpoints[model_idx]['epoch']:02d}.pth"
    checkpoint_path = config.get_final_models_dir() / model_name
    
    logger.info(f"\n  Model #{model_idx+1}/{TOP_K_MODELS}: {model_name}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model_scores.append({
        'model_idx': model_idx,
        'gen_score': best_checkpoints[model_idx]['gen_score'],
        'fold': best_checkpoints[model_idx]['fold'],
        'epoch': best_checkpoints[model_idx]['epoch']
    })
    
    # Load model
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    # Generate predictions
    model_predictions = []
    
    with torch.no_grad():
        for batch_idx, images in enumerate(test_loader):
            images = images.to(DEVICE)
            
            # Forward pass
            outputs = model(images)
            
            # Get probabilities
            probs = torch.softmax(outputs, dim=1)
            
            # Store probabilities
            model_predictions.append(probs.detach().cpu().numpy())
            
            if (batch_idx + 1) % 20 == 0:
                logger.info(f"    Batch {batch_idx + 1}/{len(test_loader)}")
    
    # Concatenate all batches for this model
    model_probs = np.vstack(model_predictions)
    test_predictions_all_models.append(model_probs)
    
    logger.info(f"  ✓ Model predictions: shape={model_probs.shape}, gen_score={best_checkpoints[model_idx]['gen_score']:.4f}")
    del model
    torch.cuda.empty_cache()

logger.info(f"\n✓ Generated predictions from all {TOP_K_MODELS} models")

In [ ]:
# STAGE 8: EXACT - Ensemble predictions with weighted average
logger.info("\nEnsembling predictions (weighted by generalization score)...")

# Stack all model predictions
all_models_probs = np.stack(test_predictions_all_models)  # Shape: (TOP_K, 3000, 10)

# Compute weights based on generalization scores
scores = np.array([best_checkpoints[i]['gen_score'] for i in range(TOP_K_MODELS)])
# Normalize scores to sum to 1
weights = scores / scores.sum()

logger.info(f"Ensemble weights (based on gen_score):")
for idx, w in enumerate(weights):
    logger.info(f"  Model #{idx+1}: {w:.4f} (gen_score={best_checkpoints[idx]['gen_score']:.4f})")

# Weighted average
ensemble_probs = (all_models_probs * weights.reshape(-1, 1, 1)).sum(axis=0)  # Shape: (3000, 10)

logger.info(f"\nEnsemble probabilities computed:")
logger.info(f"  Shape: {ensemble_probs.shape}")
logger.info(f"  Mean per sample sums to 1.0? {np.allclose(ensemble_probs.sum(axis=1), 1.0)}")

# Get final predictions
final_predictions = ensemble_probs.argmax(axis=1)  # Shape: (3000,)

logger.info(f"Final predictions computed:")
logger.info(f"  Shape: {final_predictions.shape}")
logger.info(f"  Class distribution: {np.bincount(final_predictions)}")

In [ ]:
# STAGE 8: EXACT - Map predictions to class names
logger.info("\nMapping predictions to class names...")

# Use the predefined class mapping (defined in Stage 1)
# Note: class_idx_to_name already created earlier
final_predictions_names = np.array([class_idx_to_name[idx] for idx in final_predictions])

logger.info(f"Predictions mapped to class names:")
for class_name, count in pd.Series(final_predictions_names).value_counts().items():
    logger.info(f"  {class_name}: {count}")

In [ ]:
# STAGE 8: EXACT - Create submission CSV
logger.info("\nCreating submission CSV...")

config.get_submission_dir().mkdir(parents=True, exist_ok=True)

# Get test image IDs
test_image_ids = test_df['ID'].values

# Create submission DataFrame
submission_df = pd.DataFrame({
    'ID': test_image_ids,
    'TARGET': final_predictions_names
})

# Save to CSV
submission_path = config.get_submission_dir() / "submission.csv"
submission_df.to_csv(submission_path, index=False)

logger.info(f"\n✓ Submission CSV created: {submission_path}")
logger.info(f"  Shape: {submission_df.shape}")
logger.info(f"  Columns: {list(submission_df.columns)}")
logger.info(f"\n  First 5 rows:")
for idx, row in submission_df.head().iterrows():
    logger.info(f"    {row['ID']}, {row['TARGET']}")

In [ ]:
# STAGE 8: EXACT - Verify submission format
logger.info("\nVerifying submission format...")

# Check 1: Correct number of rows
assert len(submission_df) == 3000, f"Wrong number of rows: {len(submission_df)}"
logger.info(f"✓ 3000 rows")

# Check 2: Correct columns
assert list(submission_df.columns) == ['ID', 'TARGET'], f"Wrong columns: {list(submission_df.columns)}"
logger.info(f"✓ Columns: ID, TARGET")

# Check 3: No missing values
assert not submission_df['ID'].isna().any(), "Missing values in ID column"
assert not submission_df['TARGET'].isna().any(), "Missing values in TARGET column"
logger.info(f"✓ No missing values")

# Check 4: All test IDs present
assert len(submission_df['ID'].unique()) == 3000, "Duplicate IDs in submission"
assert set(submission_df['ID'].values) == set(test_df['ID'].values), "ID mismatch"
logger.info(f"✓ All test IDs present, no duplicates")

# Check 5: All predictions are valid class names
valid_classes = set(class_idx_to_name.values())
invalid_preds = ~submission_df['TARGET'].isin(valid_classes)
assert not invalid_preds.any(), f"Invalid class predictions found"
logger.info(f"✓ All predictions are valid class names")

# Check 6: File exists and is readable
assert submission_path.exists(), f"Submission file not found"
file_size_kb = submission_path.stat().st_size / 1e3
logger.info(f"✓ File saved: {file_size_kb:.1f} KB")

logger.info(f"\n{'='*60}")
logger.info(f"✓✓✓ SUBMISSION VERIFIED AND READY FOR KAGGLE ✓✓✓")
logger.info(f"{'='*60}")
logger.info(f"\nNext step: Upload {submission_path} to Kaggle competition")
logger.info(f"\nPipeline Summary:")
logger.info(f"  - Top {TOP_K_MODELS} models: {[best_checkpoints[i]['fold'] for i in range(TOP_K_MODELS)]}")
logger.info(f"  - Ensemble method: Weighted average (by gen_score)")
logger.info(f"  - Space saved: ~{(NUM_FOLDS * EPOCHS - TOP_K_MODELS) * 110 / 1024:.1f} GB")

In [ ]:
# STAGE 8: EXACT - Save metadata and confidence scores
logger.info("\nSaving metadata and confidence scores...")

metadata = {
    'timestamp': str(pd.Timestamp.now()),
    'num_test_samples': len(submission_df),
    'num_classes': NUM_CLASSES,
    'ensemble_strategy': f'Weighted average of top {TOP_K_MODELS} models',
    'model_architecture': MODEL_NAME,
    'model_source': 'pytorch-image-models (timm)',
    'num_folds_trained': NUM_FOLDS,
    'epochs_per_fold': EPOCHS,
    'total_checkpoints_trained': NUM_FOLDS * EPOCHS,
    'top_k_models_kept': TOP_K_MODELS,
    'space_saved_gb': (NUM_FOLDS * EPOCHS - TOP_K_MODELS) * 110 / 1024,
    'top_models': [
        {
            'rank': r+1,
            'fold': best_checkpoints[r]['fold'],
            'epoch': best_checkpoints[r]['epoch'],
            'gen_score': float(best_checkpoints[r]['gen_score']),
            'val_acc': float(best_checkpoints[r]['val_acc']),
            'train_acc': float(best_checkpoints[r]['train_acc']),
            'weight': float(weights[r])
        }
        for r in range(TOP_K_MODELS)
    ]
}

metadata_path = config.get_submission_dir() / "submission_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"✓ Saved submission_metadata.json")
logger.info(f"\n✓✓✓ PIPELINE COMPLETE ✓✓✓")
logger.info(f"Config path: {config.resolve_path('pipeline_kaggle.toml')}")
logger.info(f"Logs: {config.get_logs_dir()}/pipeline.log")